# Traffic Accidents: Data Collection and Exploration

**Course:** CSC17104 - Programming for Data Science  
**Group:** Group 14 - Tam Khuyen  

This notebook documents the dataset provenance, examines its structure and quality, performs shared baseline preprocessing, and identifies limitations and patterns that guide the project's analytical questions.

## 1. Data collection

### 1.1 Subject and real-world context

The dataset describes reported traffic crashes in Chicago. Each row represents one recorded crash and contains information about the crash time, traffic controls, weather and lighting, roadway conditions, collision type, reported cause, vehicle count, damage category, and injuries.

### 1.2 Source and provenance

- **Kaggle dataset:** [Traffic Accidents by Oktay Ordekci](https://www.kaggle.com/datasets/oktayrdeki/traffic-accidents)
- **Official source context:** [Traffic Crashes - Crashes, City of Chicago Data Portal](https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if/about_data)
- **Local raw file:** `data/raw/traffic_accidents.csv`
- **Snapshot size:** 209,306 rows and 24 columns
- **Observed date range:** 2013-03-03 to 2025-01-18

The Kaggle publisher and the official data provider should both be credited. The exact publication or update date of the Kaggle version used by the group must be copied from its version metadata before final submission because it is not stored in the CSV file.

### 1.3 License and attribution

Kaggle labels the dataset license as `Other (specified in description)`, while the official Chicago portal refers users to its Terms of Use. The group must verify and record the exact applicable reuse and attribution terms in the final README. Until that verification is complete, this project treats the data as public data approved for educational analysis and attributes both the Kaggle publisher and the City of Chicago.

### 1.4 Collection method and population

The official Chicago dataset is based on traffic-crash reports recorded in the Chicago Police Department's electronic crash-reporting system. Reports may be completed by responding officers at the scene or self-reported at a police district and can later be amended. The population represented is reported crashes on streets within Chicago city limits and CPD jurisdiction; crashes handled by another agency may be excluded.

### 1.5 Known limitations and potential biases

- Coverage before September 2017 is not citywide, and the 2025 data are only a partial year.
- Weather, lighting, road condition, and contributory cause may reflect the reporting officer's best available assessment.
- Some crashes are self-reported, which may create reporting differences.
- The dataset contains only reported crashes and no traffic-exposure denominator, so it cannot directly estimate a driver's probability of crashing.
- Missing information is often encoded as `UNKNOWN`, `NOT APPLICABLE`, or `UNABLE TO DETERMINE` instead of a conventional null value.
- This Kaggle extract lacks a unique crash identifier and detailed location fields, limiting duplicate verification and spatial analysis.

### 1.6 Motivation

The dataset is large, real-world, and contains temporal, categorical, and injury-related variables. It supports reproducible data cleaning, statistical comparisons, uncertainty quantification, robustness checks, external-data merging, and a time-aware machine-learning task related to traffic safety.

### 1.7 Notebook setup

The following cells import the required libraries, resolve project-relative paths, load the untouched raw CSV, and display a small sample. The raw file is never overwritten.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "traffic_accidents.csv"

df = pd.read_csv(RAW_DATA_PATH)

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
df.head()

Number of rows: 209,306
Number of columns: 24


,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
0,07/29/2023 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,13,7,7
1,08/13/2023 12:11:00 AM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,0,1,8
2,12/09/2021 10:30:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,10,5,12
3,08/09/2023 07:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,5.0,0.0,0.0,5.0,0.0,0.0,19,4,8
4,08/19/2023 02:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,14,7,8


## 2. Initial data inspection

This section examines the dataset structure, data types, missing values,
duplicates, placeholder values, and basic distributions before cleaning. Cleaning decisions are made only after the initial audit is documented.

### 2.1 Dataset overview and data dictionary

One row represents one recorded traffic crash. The table below summarizes the 24 raw columns before any conversion or feature engineering.

| Column | Meaning | Raw type |
|---|---|---|
| `crash_date` | Reported date and time of the crash | string |
| `traffic_control_device` | Traffic-control device present | string |
| `weather_condition` | Reported weather condition | string |
| `lighting_condition` | Reported lighting condition | string |
| `first_crash_type` | Type of the first collision | string |
| `trafficway_type` | Roadway or intersection configuration | string |
| `alignment` | Road alignment and grade | string |
| `roadway_surface_cond` | Reported road-surface condition | string |
| `road_defect` | Reported road defect | string |
| `crash_type` | Broad injury/tow or drive-away category | string |
| `intersection_related_i` | Whether the crash is intersection-related | string |
| `damage` | Estimated property-damage category | string |
| `prim_contributory_cause` | Primary reported contributory cause | string |
| `num_units` | Number of traffic units involved | integer |
| `most_severe_injury` | Most severe injury recorded in the crash | string |
| `injuries_total` | Total number of injured people | float representing a count |
| `injuries_fatal` | Number of fatal injuries | float representing a count |
| `injuries_incapacitating` | Number of incapacitating injuries | float representing a count |
| `injuries_non_incapacitating` | Number of non-incapacitating injuries | float representing a count |
| `injuries_reported_not_evident` | Number of reported injuries not visibly evident | float representing a count |
| `injuries_no_indication` | Number of people with no injury indication | float representing a count |
| `crash_hour` | Hour of day, from 0 to 23 | integer |
| `crash_day_of_week` | Day code, where Sunday is 1 and Saturday is 7 | integer |
| `crash_month` | Month number, from 1 to 12 | integer |

In [3]:
overview = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=False)
})

display(overview)

,data_type,non_null_count,missing_count,missing_percentage,unique_count
crash_date,str,209306,0,0.0,189087
traffic_control_device,str,209306,0,0.0,19
weather_condition,str,209306,0,0.0,12
lighting_condition,str,209306,0,0.0,6
first_crash_type,str,209306,0,0.0,18
trafficway_type,str,209306,0,0.0,20
alignment,str,209306,0,0.0,6
roadway_surface_cond,str,209306,0,0.0,7
road_defect,str,209306,0,0.0,7
crash_type,str,209306,0,0.0,2


### 2.2 Duplicate and blank rows

Exact duplicate rows and empty strings are checked before cleaning. Duplicate removal is not automatic at this stage because the extract does not include a unique crash identifier; the number and proportion affected must first be reported.

In [4]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = duplicate_count / len(df) * 100

print(f"Duplicated rows: {duplicate_count:,}")
print(f"Duplicated percentage: {duplicate_percentage:.4f}%")

Duplicated rows: 31
Duplicated percentage: 0.0148%


In [5]:
duplicate_rows = df[df.duplicated(keep=False)]

display(duplicate_rows.head(20))

,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
5303,02/14/2020 03:48:00 PM,STOP SIGN/FLASHER,CLEAR,DAYLIGHT,PEDESTRIAN,NOT DIVIDED,STRAIGHT AND LEVEL,SNOW OR SLUSH,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,INCAPACITATING INJURY,2.0,0.0,1.0,0.0,1.0,0.0,15,6,2
7930,07/17/2023 04:05:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,16,2,7
10478,02/25/2020 08:10:00 AM,TRAFFIC SIGNAL,SNOW,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,WET,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,2.0,0.0,0.0,2.0,0.0,3.0,8,3,2
16323,02/14/2022 11:00:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,11,2,2
16680,09/06/2023 04:27:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,2.0,0.0,0.0,2.0,0.0,0.0,16,4,9
18523,07/17/2024 09:15:00 PM,TRAFFIC SIGNAL,CLEAR,DUSK,SIDESWIPE SAME DIRECTION,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,21,4,7
25351,05/19/2020 11:42:00 PM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",ANGLE,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,3.0,0.0,0.0,3.0,0.0,1.0,23,3,5
31704,08/31/2018 02:30:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,14,6,8
33097,04/02/2018 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,13,2,4
33320,12/22/2018 01:40:00 PM,STOP SIGN/FLASHER,CLEAR,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,13,7,12


In [6]:
object_columns = df.select_dtypes(include="object").columns

blank_counts = pd.Series({
    column: (
        df[column].notna()
        & df[column].astype("string").str.strip().eq("")
    ).sum()
    for column in object_columns
})

blank_counts = blank_counts[blank_counts > 0]

display(blank_counts.to_frame("blank_count"))

/tmp/ipykernel_6583/2707586074.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = df.select_dtypes(include="object").columns


,blank_count


### 2.3 Missing and placeholder values

The CSV may contain no conventional nulls while still encoding unavailable information as text. Therefore, `UNKNOWN`, `NOT APPLICABLE`, and `UNABLE TO DETERMINE` are counted by column and retained as explicit categories during shared preprocessing. Question-specific analyses may exclude or regroup them only with a stated justification and a robustness check.

In [7]:
placeholder_values = {
    "UNKNOWN",
    "NOT APPLICABLE",
    "UNABLE TO DETERMINE"
}

placeholder_records = []

for column in object_columns:
    normalized = df[column].astype("string").str.strip().str.upper()
    counts = normalized.value_counts()

    for value in placeholder_values:
        count = counts.get(value, 0)

        if count > 0:
            placeholder_records.append({
                "column": column,
                "value": value,
                "count": count,
                "percentage": round(count / len(df) * 100, 2)
            })

placeholder_summary = pd.DataFrame(placeholder_records)

display(
    placeholder_summary.sort_values(
        ["column", "count"],
        ascending=[True, False]
    )
)

,column,value,count,percentage
2,lighting_condition,UNKNOWN,4336,2.07
6,prim_contributory_cause,UNABLE TO DETERMINE,58316,27.86
7,prim_contributory_cause,NOT APPLICABLE,5241,2.50
5,road_defect,UNKNOWN,34426,16.45
4,roadway_surface_cond,UNKNOWN,12509,5.98
0,traffic_control_device,UNKNOWN,4455,2.13
3,trafficway_type,UNKNOWN,1060,0.51
1,weather_condition,UNKNOWN,6534,3.12


### 2.4 Numerical variables

The numerical summary is used to verify ranges and identify values that require closer inspection. Large injury counts or numbers of units are not removed merely because they are statistical outliers; they may represent genuine multi-vehicle or high-severity crashes.

In [8]:
numeric_summary = df.describe(include="number").T

display(numeric_summary)

,count,mean,std,min,25%,50%,75%,max
num_units,209306.0,2.063300,0.396012,1.0,2.0,2.0,2.0,11.0
injuries_total,209306.0,0.382717,0.799720,0.0,0.0,0.0,1.0,21.0
injuries_fatal,209306.0,0.001859,0.047502,0.0,0.0,0.0,0.0,3.0
injuries_incapacitating,209306.0,0.038102,0.233964,0.0,0.0,0.0,0.0,7.0
injuries_non_incapacitating,209306.0,0.221241,0.614960,0.0,0.0,0.0,0.0,21.0
injuries_reported_not_evident,209306.0,0.121516,0.450865,0.0,0.0,0.0,0.0,15.0
injuries_no_indication,209306.0,2.244002,1.241175,0.0,2.0,2.0,3.0,49.0
crash_hour,209306.0,13.373047,5.603830,0.0,9.0,14.0,17.0,23.0
crash_day_of_week,209306.0,4.144024,1.966864,1.0,2.0,4.0,6.0,7.0
crash_month,209306.0,6.771822,3.427593,1.0,4.0,7.0,10.0,12.0


### 2.5 Categorical variables

The number of distinct values, dominant labels, rare categories, and class imbalance are examined for the categorical columns. The detailed frequency tables help determine whether later analyses should preserve, exclude, or combine particular categories.

In [9]:
categorical_summary = df.describe(include="object").T

display(categorical_summary)

/tmp/ipykernel_6583/275919698.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_summary = df.describe(include="object").T


,count,unique,top,freq
crash_date,209306,189087,12/29/2020 05:00:00 PM,10
traffic_control_device,209306,19,TRAFFIC SIGNAL,123944
weather_condition,209306,12,CLEAR,164700
lighting_condition,209306,6,DAYLIGHT,134109
first_crash_type,209306,18,TURNING,64157
trafficway_type,209306,20,NOT DIVIDED,77753
alignment,209306,6,STRAIGHT AND LEVEL,204590
roadway_surface_cond,209306,7,DRY,155905
road_defect,209306,7,NO DEFECTS,171730
crash_type,209306,2,NO INJURY / DRIVE AWAY,117376


In [10]:
for column in object_columns:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).head(10))


--- crash_date ---


crash_date
12/29/2020 05:00:00 PM    10
02/17/2022 03:30:00 PM     8
06/17/2019 04:30:00 PM     6
11/26/2018 08:30:00 AM     6
09/27/2018 08:30:00 AM     6
01/12/2019 02:30:00 PM     6
06/05/2021 01:00:00 PM     6
02/26/2020 07:45:00 AM     6
09/11/2019 04:30:00 PM     6
01/24/2019 04:00:00 PM     6
Name: count, dtype: int64


--- traffic_control_device ---


traffic_control_device
TRAFFIC SIGNAL              123944
STOP SIGN/FLASHER            49139
NO CONTROLS                  29508
UNKNOWN                       4455
OTHER                          670
YIELD                          468
PEDESTRIAN CROSSING SIGN       247
OTHER REG. SIGN                181
LANE USE MARKING               153
FLASHING CONTROL SIGNAL        150
Name: count, dtype: int64


--- weather_condition ---


weather_condition
CLEAR                    164700
RAIN                      21703
CLOUDY/OVERCAST            7533
SNOW                       6871
UNKNOWN                    6534
OTHER                       627
FREEZING RAIN/DRIZZLE       510
FOG/SMOKE/HAZE              360
SLEET/HAIL                  308
BLOWING SNOW                127
Name: count, dtype: int64

... (18 outputs left out)


--- prim_contributory_cause ---


prim_contributory_cause
UNABLE TO DETERMINE                       58316
FAILING TO YIELD RIGHT-OF-WAY             42914
FOLLOWING TOO CLOSELY                     19084
DISREGARDING TRAFFIC SIGNALS              14591
IMPROPER TURNING/NO SIGNAL                12643
FAILING TO REDUCE SPEED TO AVOID CRASH    10676
IMPROPER OVERTAKING/PASSING                8302
DISREGARDING STOP SIGN                     6749
IMPROPER LANE USAGE                        6462
NOT APPLICABLE                             5241
Name: count, dtype: int64


--- most_severe_injury ---


most_severe_injury
NO INDICATION OF INJURY     154789
NONINCAPACITATING INJURY     31527
REPORTED, NOT EVIDENT        16075
INCAPACITATING INJURY         6564
FATAL                          351
Name: count, dtype: int64

### 2.6 Date and time validation

The date strings are parsed with an explicit format. The derived hour, weekday, and month are then compared with the corresponding raw columns to detect inconsistent records before the date field is converted permanently.

In [11]:
parsed_crash_date = pd.to_datetime(
    df["crash_date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

print(f"Invalid dates: {parsed_crash_date.isna().sum():,}")
print(f"Earliest date: {parsed_crash_date.min()}")
print(f"Latest date: {parsed_crash_date.max()}")

Invalid dates: 0
Earliest date: 2013-03-03 16:48:00
Latest date: 2025-01-18 00:17:00


In [12]:
expected_day_of_week = (
    (parsed_crash_date.dt.dayofweek + 1) % 7
) + 1

consistency_check = pd.Series({
    "hour_mismatches": (
        df["crash_hour"] != parsed_crash_date.dt.hour
    ).sum(),

    "day_of_week_mismatches": (
        df["crash_day_of_week"] != expected_day_of_week
    ).sum(),

    "month_mismatches": (
        df["crash_month"] != parsed_crash_date.dt.month
    ).sum()
})

display(consistency_check.to_frame("mismatch_count"))

,mismatch_count
hour_mismatches,0
day_of_week_mismatches,0
month_mismatches,0


### 2.7 Initial inspection findings

- The raw extract contains 209,306 rows and 24 columns.
- No conventional null values or blank strings were found.
- There are 31 exact duplicate rows (0.0148% of the data). Because no unique crash identifier is available, they are treated as probable duplicates and their removal should be checked in a robustness analysis.
- Unavailable information is mainly encoded through text. In particular, `UNABLE TO DETERMINE` is common in `prim_contributory_cause`, and `UNKNOWN` is common in several road and environmental variables. Removing all such rows would cause substantial information loss.
- All `crash_date` values parse successfully. The stored hour, weekday, and month agree with the parsed timestamp for every row.
- Numerical ranges are structurally valid: hours are 0-23, weekday codes are 1-7, month codes are 1-12, and `num_units` is always at least one. Large injury counts are retained as potentially genuine events.
- The observed date range includes incomplete early coverage and a partial 2025. Full-year temporal comparisons should therefore focus on 2018-2024 unless a different window is justified.

## 3. Data cleaning and preprocessing

The initial inspection found no conventional missing values, blank strings,
invalid dates, or inconsistencies among the time and injury columns.

The dataset contains 31 exact duplicate rows. These duplicates are removed
from the processed dataset because every available attribute is identical.
However, because the dataset does not provide a unique crash identifier,
the effect of retaining them will be considered in a robustness check.

Values such as `UNKNOWN`, `NOT APPLICABLE`, and `UNABLE TO DETERMINE`
are preserved as explicit categories. They represent unavailable or
undetermined information and will be handled separately for each question.

In [13]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_clean

clean_df = load_clean(RAW_DATA_PATH)

Rows before removing duplicates: 209,306
Rows after removing duplicates: 209,275
Rows removed: 31


In [14]:
cleaning_summary = pd.Series({
    "number_of_rows": len(clean_df),
    "number_of_columns": clean_df.shape[1],
    "duplicated_rows": clean_df.duplicated().sum(),
    "missing_values": clean_df.isna().sum().sum(),
    "earliest_date": clean_df["crash_date"].min(),
    "latest_date": clean_df["crash_date"].max()
})

display(cleaning_summary)

number_of_rows                    209275
number_of_columns                     29
duplicated_rows                        0
missing_values                         0
earliest_date        2013-03-03 16:48:00
latest_date          2025-01-18 00:17:00
dtype: object

In [15]:
PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "traffic_accidents_clean.csv"
)

clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Processed data saved to: {PROCESSED_DATA_PATH}")

Processed data saved to: /home/doantoi/p4ds/traffic_project/data/processed/traffic_accidents_clean.csv


## 4. Current preprocessing output and next exploration steps

The shared baseline preprocessing produces 209,275 rows and 29 columns. It removes 31 exact duplicate rows, parses `crash_date`, converts injury counts to integers, and creates `crash_year`, `has_injury`, `has_severe_injury`, `is_weekend`, and `time_period`. Placeholder categories are preserved. The reproducible result is written to `data/processed/traffic_accidents_clean.csv`; the processed file is not committed to Git.

Before this exploration notebook is considered complete, the following analyses still need to be added:

1. Yearly and monthly record counts, with incomplete coverage explicitly marked.
2. Distributions and outlier checks for the numerical variables most relevant to the questions.
3. Balance plots for important categorical variables rather than one plot for every column.
4. Relationships such as weather by injury status, lighting by injury status, collision type by severe injury, and intersection status by severity.
5. A final summary of 3-5 key observations, serious data-quality issues, preprocessing decisions, and patterns that motivate Q1-Q6.